# Edge, GPU monitoring, Netron, TFLite/LiteRT i TensorFlow.js

Noteboook prezentuje rózne środowiska uruchomieniowe dla modeli ML: GPU, urządzenia edge, IoT oraz uruchamianie w przeglądarce.


In [1]:
import tensorflow as tf
from pathlib import Path

(x_train, y_train), _ = tf.keras.datasets.mnist.load_data()
x_train = x_train.astype("float32") / 255.0

model = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(28, 28)),
    tf.keras.layers.Flatten(),
    tf.keras.layers.Dense(32, activation="relu"),
    tf.keras.layers.Dense(10, activation="softmax"),
])
model.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"])
model.fit(x_train[:5000], y_train[:5000], epochs=1, batch_size=128)

Path("models").mkdir(exist_ok=True)
model.save("models/mnist_small.keras")
print("Saved Keras model: models/mnist_small.keras")


I0000 00:00:1781763996.227671   44577 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1781763996.268784   44577 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX_VNNI AVX_VNNI_INT8 AVX_NE_CONVERT FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1781763998.301475   44577 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


11490434/11490434 ━━━━━━━━━━━━━━━━━━━━ 2s 0us/step


W0000 00:00:1781764002.016490   44625 cuda_executor.cc:1755] Failed to determine cuDNN version (Note that this is expected if the application doesn't link the cuDNN plugin): INTERNAL: cuDNN error: CUDNN_STATUS_INTERNAL_ERROR
W0000 00:00:1781764002.251148   44577 gpu_device.cc:2365] Cannot dlopen some GPU libraries. Please make sure the missing libraries mentioned above are installed properly if you would like to use GPU. Follow the guide at https://www.tensorflow.org/install/gpu for how to download and setup the required libraries for your platform.
Skipping registering GPU devices...


40/40 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.5126 - loss: 1.6728
Saved Keras model: models/mnist_small.keras


## Netron

Netron pozwala szybko obejrzeć architekturę modelu.

Setup:
1. Wejść na `https://netron.app` i przeciągnąć plik `models/mnist_small.keras`.
2. Zainstalować lokalnie:

```bash
pip install netron
netron models/mnist_small.keras
```


In [2]:
# Konwersja do TensorFlow Lite / LiteRT
converter = tf.lite.TFLiteConverter.from_keras_model(model)
tflite_model = converter.convert()

with open("models/mnist_small.tflite", "wb") as f:
    f.write(tflite_model)

print("TFLite model size:", len(tflite_model) / 1024, "KB")


INFO:tensorflow:Assets written to: /tmp/tmpbi7xvus4/assets


INFO:tensorflow:Assets written to: /tmp/tmpbi7xvus4/assets


Saved artifact at '/tmp/tmpbi7xvus4'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 28, 28), dtype=tf.float32, name='keras_tensor')
Output Type:
  TensorSpec(shape=(None, 10), dtype=tf.float32, name=None)
Captures:
  129524930697296: TensorSpec(shape=(), dtype=tf.resource, name=None)
  129524930698256: TensorSpec(shape=(), dtype=tf.resource, name=None)
  129524930697872: TensorSpec(shape=(), dtype=tf.resource, name=None)
  129524930698640: TensorSpec(shape=(), dtype=tf.resource, name=None)
TFLite model size: 102.11328125 KB


W0000 00:00:1781764003.309040   44577 tf_tfl_flatbuffer_helpers.cc:364] Ignored output_format.
W0000 00:00:1781764003.309062   44577 tf_tfl_flatbuffer_helpers.cc:367] Ignored drop_control_dependency.
I0000 00:00:1781764003.309287   44577 reader.cc:83] Reading SavedModel from: /tmp/tmpbi7xvus4
I0000 00:00:1781764003.309535   44577 reader.cc:52] Reading meta graph with tags { serve }
I0000 00:00:1781764003.309540   44577 reader.cc:147] Reading SavedModel debug info (if present) from: /tmp/tmpbi7xvus4
I0000 00:00:1781764003.312333   44577 mlir_graph_optimization_pass.cc:437] MLIR V1 optimization pass is not enabled
I0000 00:00:1781764003.312675   44577 loader.cc:236] Restoring SavedModel bundle.
I0000 00:00:1781764003.330570   44577 loader.cc:220] Running initialization op on SavedModel bundle at path: /tmp/tmpbi7xvus4
I0000 00:00:1781764003.335813   44577 loader.cc:471] SavedModel load for tags { serve }; Status: success: OK. Took 26533 microseconds.
I0000 00:00:1781764003.345115   44577

## TensorFlow.js

TensorFlow.js pozwala uruchamiać modele w przeglądarce lub Node.js.

Eksport można wykonać narzędziem `tensorflowjs_converter`:

```bash
pip install tensorflowjs
tensorflowjs_converter --input_format=keras models/mnist_small.keras tfjs_model/
```

W przeglądarce model można załadować przez `tf.loadLayersModel(...)`.


## nvidia-smi — monitoring GPU

Podstawowe komendy do uruchomienia lokalnie:

```bash
nvidia-smi
nvidia-smi --loop=1
nvidia-smi dmon
nvidia-smi pmon
```


Jakie zasoby mozna monitorować w nvidia-smi?
- wykorzystanie GPU;
- użycie pamięci;
- procesy korzystające z GPU;
- temperatura i pobór mocy;
- czy model faktycznie używa GPU;
- czy bottleneck jest w GPU, CPU, dysku, dataloaderze czy batch size.


In [3]:
# Sprawdzenie GPU z poziomu TensorFlow
import tensorflow as tf
print(tf.config.list_physical_devices("GPU"))


[]


## Podsumowanie

- TFLite/LiteRT jest używane dla edge i on-device inference: telefon, IoT, embedded, urządzenia przemysłowe.
- TensorFlow.js jest przydatne, gdy model ma działać bezpośrednio w przeglądarce.
- Netron pomaga zrozumieć strukturę modelu przed deploymentem.
- `nvidia-smi` jest najprostszym narzędziem do szybkiego sprawdzenia, czy model naprawdę korzysta z GPU i ile zasobów zużywa.
